In [1]:
# Install Unsloth from source
!git clone https://github.com/billishyahao/unsloth.git && cd unsloth && git checkout billhe/rocm &&  pip install .
!pip install unsloth_zoo==2025.3.17
# Install ROCm Bitsandbytes from source 
!git clone --recurse https://github.com/ROCm/bitsandbytes && cd bitsandbytes && git checkout rocm_enabled_multi_backend && pip install -r requirements-dev.txt && cmake -DCOMPUTE_BACKEND=hip -S . && make -j  && pip install .
# This notebook is verified under unsloth==2025.3.19 unsloth_zoo==2025.3.17 bitsandbytes==0.43.3.dev0

fatal: destination path 'unsloth' already exists and is not an empty directory.
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/HipMarker-1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/rpdTracer-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
fatal: destination path 'bitsandbytes' already exists and is not an empty directory.


In [2]:
# Verify the installation and version of the required libraries
!pip list | grep unsloth

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/HipMarker-1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/rpdTracer-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
unsloth                           2025.3.19
unsloth_zoo                       2025.3.17


In [3]:
from huggingface_hub import notebook_login, HfApi

# Prompt the user to log in
notebook_login()


In [4]:
from huggingface_hub import HfApi

try:
    api = HfApi()
    user_info = api.whoami()
    print(f"Token validated successfully! Logged in as: {user_info['name']}")
except Exception as e:
    print(f"Token validation failed. Error: {e}")

Token validated successfully! Logged in as: Erebus007


In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 
lora_rank = 32 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # Best for MI300X stability
    fast_inference = True,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, 
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

g++ (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:154: UserWarning: Unsloth: Running `ldconfig /usr/lib64-nvidia` to link CUDA.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:188: UserWarning: Unsloth: CUDA is not linked properly.
Try running `python -m bitsandbytes` then `python -m xformers.info`
We tried running `ldconfig /usr/lib64-nvidia` ourselves, but it didn't work.
You need to run in your terminal `sudo ldconfig /usr/lib64-nvidia` yourself, then import Unsloth.
Also try `sudo ldconfig /usr/local/cuda-xx.x` - find the latest cuda version.
Unsloth will still run for now, but maybe it might crash - let's hope it works!
  warnings.warn(


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-09 23:48:27 [__init__.py:256] Automatically detected platform rocm.
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.1. vLLM: 0.7.4.dev388+g51641aaa7.rocm631.
   \\   /|    AMD Instinct MI300X VF. Num GPUs = 1. Max memory: 191.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0a0+git6c0e746. CUDA: 9.4. CUDA Toolkit: None. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Llama-3.2-3B-Instruct with actual GPU utilization = 49.91%
Unsloth: Your GPU has CUDA compute capability 9.4 with VRAM = 191.69 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 400.
Unsloth: vLLM's KV Cache 

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-09 23:49:02 [loader.py:429] Loading weights took 2.33 seconds
INFO 05-09 23:49:02 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 05-09 23:49:02 [model_runner.py:1151] Model loading took 7.1582 GB and 18.037362 seconds
INFO 05-09 23:49:22 [worker.py:295] Memory profiling takes 19.69 seconds
INFO 05-09 23:49:22 [worker.py:295] the current vLLM instance can use total_gpu_memory (191.69GiB) x gpu_memory_utilization (0.50) = 95.67GiB
INFO 05-09 23:49:22 [worker.py:295] model weights take 7.16GiB; non_torch_memory takes 1.00GiB; PyTorch activation peak memory takes 1.83GiB; the rest of the memory reserved for KV Cache is 85.69GiB.
INFO 05-09 23:49:22 [executor_base.py:111] # rocm blocks: 50142, # CPU blocks: 3510
INFO 05-09 23:49:22 [executor_base.py:116] Maximum concurrency for 1024 tokens per request: 783.47x
INFO 05-09 23:49:23 [model_runner.py:1447] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in ea

Capturing CUDA graph shapes: 100%|█████████████████████████████████████████████████████████████████████████████████████| 53/53 [00:17<00:00,  3.07it/s]

INFO 05-09 23:49:41 [model_runner.py:1575] Graph capturing finished in 17 secs, took 0.35 GiB
INFO 05-09 23:49:41 [llm.py:244] init engine (profile, create kv cache, warmup model) took 38.44 seconds



Unsloth 2025.3.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
import re
from datasets import load_dataset, Dataset

SYSTEM_PROMPT = """Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>"""

# Load NCERT Dataset
def get_ncert_questions(split = "train") -> Dataset:
    data = load_dataset("KadamParth/Ncert_dataset", split = split)
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['Question']}
        ],
        'answer': x['Answer']
    })
    return data

dataset = get_ncert_questions()

# Reward 1: Checks for correct XML tags
def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [c[0]["content"] for c in completions]
    rewards = []
    for content in contents:
        count = 0.0
        if content.count("<reasoning>") == 1: count += 0.125
        if content.count("</reasoning>") == 1: count += 0.125
        if content.count("<answer>") == 1: count += 0.125
        if content.count("</answer>") == 1: count += 0.125
        rewards.append(count)
    return rewards

# Reward 2: Soft format check
def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [c[0]["content"] for c in completions]
    return [0.5 if re.search(pattern, r, re.DOTALL) else 0.0 for r in responses]

# Reward 3: Factual correctness against dataset
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    for r, a in zip(responses, answer):
        match = re.search(r"<answer>(.*?)</answer>", r, re.DOTALL)
        extracted = match.group(1).strip().lower() if match else r.lower()
        rewards.append(2.0 if a.strip().lower() in extracted else 0.0)
    return rewards

README.md:   0%|          | 0.00/200 [00:00<?, ?B/s]

NCERT_Dataset.csv:   0%|          | 0.00/107M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120406 [00:00<?, ? examples/s]

Map:   0%|          | 0/120406 [00:00<?, ? examples/s]

In [7]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate = 5e-6,
    lr_scheduler_type = "cosine",
    bf16 = True,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 8, # Uses MI300X VRAM for better self-comparison
    max_prompt_length = 512,
    max_completion_length = 1024,
    max_steps = 250, # Set to -1 for full training
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


In [9]:
import re

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Rewards responses that follow the XML format strictly with no extra text."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>$"
    responses = [c[0]["content"] for c in completions]
    return [0.5 if re.match(pattern, r, re.DOTALL) else 0.0 for r in responses]

def int_reward_func(completions, **kwargs) -> list[float]:
    """Rewards answers that contain numbers (common in NCERT math/physics)."""
    responses = [c[0]["content"] for c in completions]
    extracted = [r.split("<answer>")[-1].split("</answer>")[0].strip() for r in responses]
    return [0.5 if any(char.isdigit() for char in a) else 0.0 for a in extracted]

In [10]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 120,406 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 48,627,712/3,261,377,536 (1.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completion_length,kl,rewards / xmlcount_reward_func,rewards / soft_format_reward_func,rewards / strict_format_reward_func,rewards / int_reward_func,rewards / correctness_reward_func
1,-0.000000,0.843750,0.430827,255.562500,0.000000,0.296875,0.046875,0.046875,0.203125,0.250000
2,0.000000,0.609375,0.455588,290.281250,0.000000,0.343750,0.109375,0.062500,0.093750,0.000000
3,0.000000,0.613281,0.278916,419.468750,0.000494,0.269531,0.031250,0.015625,0.296875,0.000000
4,0.000000,0.515625,0.324280,243.906250,0.000600,0.328125,0.078125,0.031250,0.078125,0.000000
5,0.000000,0.398438,0.347056,357.781250,0.000456,0.195312,0.046875,0.015625,0.140625,0.000000
6,0.000000,0.828125,0.343429,189.875000,0.000471,0.343750,0.078125,0.062500,0.343750,0.000000
7,0.000000,0.703125,0.374292,319.906250,0.000491,0.328125,0.078125,0.031250,0.265625,0.000000
8,0.000000,0.695312,0.315303,352.375000,0.000460,0.320312,0.046875,0.015625,0.312500,0.000000
9,0.000000,0.695312,0.267910,280.125000,0.000487,0.304688,0.031250,0.000000,0.359375,0.000000
10,0.000000,0.769531,0.564447,277.250000,0.000560,0.332031,0.125000,0.078125,0.234375,0.000000


TrainOutput(global_step=250, training_loss=0.0011602064412436447, metrics={'train_runtime': 3333.9872, 'train_samples_per_second': 2.4, 'train_steps_per_second': 0.075, 'total_flos': 0.0, 'train_loss': 0.0011602064412436447})

In [11]:
# Save the model to GGUF for your 4GB Mobile Device
model.save_pretrained_gguf(
    "ncert_brawler_q4", 
    tokenizer, 
    quantization_method = "q4_k_m", # Optimized for 4GB RAM
)

# Optional: Also save a version for Hugging Face if you want to share it
# model.push_to_hub_gguf("your_username/ncert-brawler-gguf", tokenizer, quantization_method = "q4_k_m")

make: Entering directory '/workspace/notebooks/llama.cpp'
make: Leaving directory '/workspace/notebooks/llama.cpp'
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info


Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")


CMAKE_BUILD_TYPE=Release
CMake Warning at CMakeLists.txt:153 (message):
  LLAMA_CURL is deprecated and will be ignored
Call Stack (most recent call first):
  CMakeLists.txt:167 (llama_option_depr)




-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.11.0
-- ggml commit:  1e5ad35d5
-- Could NOT find OpenSSL, try to set the path to OpenSSL root folder in the system variable OPENSSL_ROOT_DIR (missing: OPENSSL_CRYPTO_LIBRARY OPENSSL_INCLUDE_DIR) 
-- Generating embedded license file for target: llama-common
-- Configuring done (0.7s)


...

-- Generating done (0.2s)
-- Build files have been written to: /workspace/notebooks/llama.cpp/build
Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 176.66 out of 235.95 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28/28 [00:00<00:00, 73.75it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at ncert_brawler_q4 into bf16 GGUF format.
The output location will be /workspace/notebooks/ncert_brawler_q4/unsloth.BF16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: ncert_brawler_q4
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:

In [ ]:
# 5:20 AM Test: See if the "Brawler" is thinking
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model) # 2x faster inference

inputs = tokenizer(
    [
        {"role": "system", "content": "Respond in the following format: <reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>"},
        {"role": "user", "content": "Explain why the sky is blue based on Class 11 Physics."}
    ],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512)
print(tokenizer.decode(outputs[0]))